In [1]:
from ucimlrepo import fetch_ucirepo 
abalone = fetch_ucirepo(id=1) 
x = abalone.data.features 
y = abalone.data.targets 

In [2]:
import pandas as pd
data = pd.concat([x, y], axis=1)
data.shape[0]

4177

In [3]:
list(data.columns)

['Sex',
 'Length',
 'Diameter',
 'Height',
 'Whole_weight',
 'Shucked_weight',
 'Viscera_weight',
 'Shell_weight',
 'Rings']

In [4]:
data.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [5]:
#What is input?
#All columns except the last one are input features. The last column is the target variable.

#What is output?
#The output is the last column, which is the target variable.

#Why output is numeric?
#The output is numeric because it represents a continuous variable, which is the number of rings in the abalone. This is a regression problem where we are trying to predict a continuous value based on the input features.

In [6]:
y = y + 1.5

In [7]:
x = x[["Length", "Diameter", "Shell_weight"]]
x.head()

,Length,Diameter,Shell_weight
0,0.455,0.365,0.150
1,0.350,0.265,0.070
2,0.530,0.420,0.210
3,0.440,0.365,0.155
4,0.330,0.255,0.055


In [8]:
# Justification:

# Feature 1: Length
# Represents the overall body size of the abalone.
# As age increases, the abalone grows longer.

# Feature 2: Diameter
# Captures the cross-sectional size.
# Helps model volume/area growth along with length.

# Feature 3: Shell_weight
# Older abalones develop thicker and heavier shells.
# This is strongly correlated with age.

In [9]:
import numpy as np
X = x.values
Y = y.values

n = X.shape[0]
split_index = int(0.8 * n)

x_train = X[:split_index]
x_test  = X[split_index:]
y_train = Y[:split_index]
y_test  = Y[split_index:]

print("X_train shape:", x_train.shape)
print("X_test shape :", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


X_train shape: (3341, 3)
X_test shape : (836, 3)
y_train shape: (3341, 1)
y_test shape : (836, 1)


In [10]:
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
x_train = (x_train - mean) / std
x_test  = (x_test  - mean) / std

In [11]:
#Why normalization is needed for learning?
# Normalization is needed because features have different scales (e.g., length in cm, weight in grams).
# Large-scale features dominate the gradient and cause unstable or slow learning.
# Scaling makes all features contribute equally, resulting in faster and more stable gradient descent convergence.

In [12]:
def forward_pass(X, w, b):
    y_hat = X @ w + b
    print("Shape X:", X.shape)
    print("Shape w:", w.shape)
    print("Shape b:", np.shape(b))
    print("Shape y_hat:", y_hat.shape)
    return y_hat

# Checkpoint:
# parameters are: weights w and bias b
# number of parameters: d weights + 1 bias  


In [13]:
def mse(y, y_hat):

    loss = np.mean((y - y_hat) ** 2)
    return loss

# Checkpoint:
# why square: ensures positive errors and penalizes large errors more
# what mistakes are expensive: large prediction errors

In [14]:
# Checkpoint:
# what gradient means in words: the direction and rate of change of the loss with respect to parameters
# why subtracting gradient reduces loss: it moves parameters in the direction that decreases the loss, leading to better predictions.

In [15]:
def grad_w(X, y, y_hat):
    N = len(y)
    dW = (2/N) * X.T @ (y_hat - y)
    return dW


def grad_b(y, y_hat):
    N = len(y)
    db = (2/N) * np.sum(y_hat - y)
    return db

# Checkpoint:
# meaning of large gradient: loss is very sensitive to parameter changes
# effect of too-large learning rate: model may diverge or oscillate

In [16]:
d = x_train.shape[1]
w = np.random.randn(d,1) * 0.01

b = 0

lr = 0.01
epochs = 500

for epoch in range(epochs):
    y_hat = x_train @ w + b
    loss = mse(y_train, y_hat)
    dW = grad_w(x_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)
    w = w - lr * dW
    b = b - lr * db

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss)


# Initial expectation:
# loss should gradually decrease

# Revised expectation after training:
# loss decreases steadily with proper learning rate

Epoch: 0 Loss: 144.31672489518533
Epoch: 50 Loss: 24.612024648134426
Epoch: 100 Loss: 9.202346023715616
Epoch: 150 Loss: 7.109374259616642
Epoch: 200 Loss: 6.792303832090361
Epoch: 250 Loss: 6.719476586089517
Epoch: 300 Loss: 6.685771498121499
Epoch: 350 Loss: 6.662447113112498
Epoch: 400 Loss: 6.644506756277459
Epoch: 450 Loss: 6.63037495297784


In [17]:
y_test_hat = x_test @ w + b

test_mse = mse(y_test, y_test_hat)

test_mae = np.mean(np.abs(y_test - y_test_hat))

print("Test MSE:", test_mse)
print("Test MAE:", test_mae)

print("\nSample Predictions:\n")

for i in range(5):
    true = y_test[i][0]
    pred = y_test_hat[i][0]
    err = abs(true - pred)

    print("True Age:", true,
          " Predicted:", pred,
          " Absolute Error:", err)
    
#Checkpoint:
#Systematic errors:
#The model tends to make larger errors for older abalones with higher ring counts.

#Observed bias:
#The model predictions appear biased toward the average age because linear regression minimizes squared error.

Test MSE: 5.094909656277842
Test MAE: 1.7326760662987382

Sample Predictions:

True Age: 13.5  Predicted: 10.814577026814298  Absolute Error: 2.685422973185702
True Age: 15.5  Predicted: 9.695079296247629  Absolute Error: 5.804920703752371
True Age: 14.5  Predicted: 10.107145746253538  Absolute Error: 4.392854253746462
True Age: 14.5  Predicted: 11.112972358772193  Absolute Error: 3.3870276412278066
True Age: 13.5  Predicted: 11.346735599639176  Absolute Error: 2.1532644003608237
